# LLM Invoke Code

In [ ]:
from dotenv import load_dotenv

from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
load_dotenv()

In [ ]:
MODEL_NAME="stepfun/step-3.5-flash:free"

In [ ]:
MODEL_NAME="minimax/minimax-m2.5:free"

llm = ChatOpenRouter(
    model=MODEL_NAME,
    temperature=0,
    # max_tokens=128
)

In [ ]:
print(llm.invoke("What is the capital of Argentina? Only answer with the Capital name.").content)

In [ ]:
print(llm.invoke("Write a haiku about Football?").content)

In [ ]:
prompt = ChatPromptTemplate.from_template("Translate to Hindi: {text}")
model = ChatOpenRouter(
    model=MODEL_NAME,
    temperature=0,
    # max_tokens=128
)

In [ ]:
chain = prompt | model
response = chain.invoke({"text": "Good Morning"})
response

In [ ]:
print(response.content)

In [ ]:
response.usage_metadata

# Image Describe Code

In [1]:
import base64
from dotenv import load_dotenv
load_dotenv()

# from app.common.utils import DATA_DIR

from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, SystemMessage
# from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import Field, BaseModel

MODEL_NAME="nvidia/nemotron-nano-12b-v2-vl:free"
IMAGE_PATH= "BYLD-values.png"

/home/rishicarter/git/AIAgents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ImageInfo(BaseModel):
    caption: str = Field(description="Brief caption describing the Image.")
    tags: list[str] = Field(description="List of 3 Keywords that describe the Image.")

In [5]:
def structured_describe():
    """Describe Image as Structured JSON output"""
    llm = ChatOpenRouter(
        model=MODEL_NAME,
        temperature=0.7
    )
    print("Setting up Data....")
    with open(IMAGE_PATH, "rb") as f:
        image_data=f.read()
        base64_string = base64.b64encode(image_data).decode('utf-8')

    msgs = [
        SystemMessage(content=f"You are an Image Analyser Assistant. "),
        HumanMessage(content=[
            {"type": "text", "text": "Provide one caption and three keywords or tags to describe the image"},
            {"type": "image", "base64": base64_string, "mime_type": "image/png"}
        ])
    ]

    chain = llm.with_structured_output(schema=ImageInfo, method="json_schema", include_raw=False, strict=True)

    # chain = llm | parser
    print("Asking LLM now....")
    response = chain.invoke(msgs)

    print(response)
    return response

In [7]:
result = structured_describe()

Setting up Data....
Asking LLM now....
caption="BYLD's Core Values: Building Leadership Through Inclusivity, Customer Centricity, and Agility" tags=['BYLD Leadership', 'Core Values', 'Inclusive Culture']


In [8]:
result

ImageInfo(caption="BYLD's Core Values: Building Leadership Through Inclusivity, Customer Centricity, and Agility", tags=['BYLD Leadership', 'Core Values', 'Inclusive Culture'])

In [ ]:
type(result)

In [11]:
dict(result)

{'caption': "BYLD's Core Values: Building Leadership Through Inclusivity, Customer Centricity, and Agility",
 'tags': ['BYLD Leadership', 'Core Values', 'Inclusive Culture']}

In [ ]:
type(result.tags),result.tags

# Caption using Blip

In [12]:
from langchain_community.document_loaders import ImageCaptionLoader

In [14]:
MODEL_NAME="nvidia/nemotron-nano-12b-v2-vl:free"
IMAGE_PATH= "BYLD-values.png"

In [15]:
with open(IMAGE_PATH, "rb") as f:
    image_data=f.read()
    base64_string = base64.b64encode(image_data).decode('utf-8')
loader = ImageCaptionLoader(images=[base64_string])

In [ ]:
loader.load()

# OPEN AI 

In [19]:
import base64
from dotenv import load_dotenv
load_dotenv()

# from app.common.utils import DATA_DIR

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
# from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import Field, BaseModel

MODEL_NAME="gpt-5-nano"
IMAGE_PATH= "BYLD-values.png"

In [23]:
class ImageInfo(BaseModel):
    caption: str = Field(description="Brief caption describing the Image.")
    tags: list[str] = Field(description="List of 3 Keywords that describe the Image.")
    summary: str = Field(description="Short Summary describing the Image.")

In [28]:
def structured_describe():
    """Describe Image as Structured JSON output"""
    llm = ChatOpenAI(
        model=MODEL_NAME,
        temperature=0.7,
        reasoning_effort="minimal"
    )
    print("Setting up Data....")
    with open(IMAGE_PATH, "rb") as f:
        image_data=f.read()
        base64_string = base64.b64encode(image_data).decode('utf-8')

    msgs = [
        SystemMessage("You are a precise Image Analysis Assistant. Be concise: caption <=20 words, summary <=50 words, 3 short tags."
    ),
        HumanMessage(content=[
            {"type": "text", "text": "Extract key visual details from the image."},
            {"type": "image", "base64": base64_string, "mime_type": "image/png"}
        ])
    ]

    chain = llm.with_structured_output(schema=ImageInfo, method="json_schema", include_raw=True, strict=True)

    # chain = llm | parser
    print("Asking LLM now....")
    response = chain.invoke(msgs)

    print(response)
    return response

In [29]:
result = structured_describe()

Setting up Data....
Asking LLM now....
{'raw': AIMessage(content='{"caption":"BYLD values circle with five core values around a central logo.","tags":["Values","Teamwork","Leadership"],"summary":"Circle diagram highlighting five core values: Inclusivity, Customer centricity, Agility, Reciprocity, Entrepreneurship, arranged around a BYLD central logo."}', additional_kwargs={'parsed': ImageInfo(caption='BYLD values circle with five core values around a central logo.', tags=['Values', 'Teamwork', 'Leadership'], summary='Circle diagram highlighting five core values: Inclusivity, Customer centricity, Agility, Reciprocity, Entrepreneurship, arranged around a BYLD central logo.'), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 731, 'total_tokens': 803, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_toke

In [26]:
dict(result)

{'caption': 'BYLD values diagram with circular flow and five value labels.',
 'tags': ['Values', 'Leadership', 'Circular diagram'],
 'summary': 'A circular diagram centered on BYLD showcasing five core values: Inclusivity, Customer centricity, Agility, Reciprocity, and Entrepreneurship, connected with numbered orange markers around the dashed circle.'}

In [35]:
dict(result["raw"])["usage_metadata"]

{'input_tokens': 731,
 'output_tokens': 72,
 'total_tokens': 803,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}